In [ ]:
%matplotlib inline

from pathlib import Path
import heapq
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
from matplotlib.collections import LineCollection

import dataset_generator as gen

BASE_DIR = Path(gen.__file__).resolve().parent
DATA_DIR = BASE_DIR / "Datasets"
DATA_DIR.mkdir(parents=True, exist_ok=True)
plt.rcParams["figure.dpi"] = 110

print("Ready! Using generator:", Path(gen.__file__).resolve())
print("Dataset output folder:", DATA_DIR.resolve())

Ready! Using generator: /Users/aayush/Desktop/DSA/Graph_Routing_Project/dataset_generator.py
Dataset output folder: /Users/aayush/Desktop/DSA/Graph_Routing_Project/simulation


In [2]:
def astar_reference(ds, query=0):
    """A* on A* datasets: return (distance, path, expanded_node_ids)."""
    coords = ds["coords"]
    s, goal, _ = ds["queries"][query]
    adjacency = {u: [] for u in coords}
    for u, v, w in ds["edges"]:
        adjacency.setdefault(u, []).append((v, w))

    if ds.get("kind") == "grid":
        def h(u):
            return abs(coords[u][0] - coords[goal][0]) + abs(coords[u][1] - coords[goal][1])
    else:
        def h(u):
            return math.dist(coords[u], coords[goal])

    g = {s: 0.0}
    parent = {}
    expanded = []
    visited = set()
    pq = [(h(s), -0.0, s)]   # larger g wins ties in f

    while pq:
        _, _, u = heapq.heappop(pq)
        if u in visited:
            continue
        visited.add(u)
        expanded.append(u)
        if u == goal:
            path = [goal]
            while path[-1] != s:
                path.append(parent[path[-1]])
            path.reverse()
            return g[goal], path, expanded
        for v, weight in adjacency.get(u, []):
            candidate = g[u] + weight
            if candidate < g.get(v, math.inf):
                g[v] = candidate
                parent[v] = u
                heapq.heappush(pq, (candidate + h(v), -candidate, v))

    return math.inf, [], expanded


def negative_cycle_nodes(n, edges):
    """Find one negative cycle using Bellman-Ford, if any."""
    dist = [0.0] * n
    parent = [-1] * n
    last_changed = -1
    for _ in range(n):
        last_changed = -1
        for u, v, w in edges:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                parent[v] = u
                last_changed = v
        if last_changed == -1:
            return []
    x = last_changed
    for _ in range(n):
        x = parent[x]
    backward = [x]
    y = parent[x]
    while y != x:
        backward.append(y)
        y = parent[y]
    return backward[::-1]


def pretty_distance(d):
    return "unreachable (∞)" if math.isinf(d) else f"{d:.4f}".rstrip("0").rstrip(".")

In [3]:

def plot_johnson(ds):
    n = len(ds["coords"])
    edges = ds["edges"]
    expected = ds.get("expected")

    show_graph = n <= 30
    show_matrix = n <= 200
    show_expected = isinstance(expected, list) and n <= 50

    count = int(show_graph) + 1 + int(show_expected)

    fig, axes = plt.subplots(
        1, count,
        figsize=(6.1 * count, 6),
        squeeze=False,
        layout="constrained"
    )

    axes = iter(axes[0])

    cycle = negative_cycle_nodes(n, edges) if n <= 400 else []

    cycle_edges = (
        {
            (cycle[i], cycle[(i + 1) % len(cycle)])
            for i in range(len(cycle))
        }
        if cycle else set()
    )

    # ----------------------------------
    # DRAW DIRECTED GRAPH
    # ----------------------------------
    if show_graph:
        ax = next(axes)

        angle = {
            i: 2 * math.pi * i / n
            for i in range(n)
        }

        pos = {
            i: (math.cos(angle[i]), math.sin(angle[i]))
            for i in range(n)
        }

        for u, v, w in edges:
            is_cycle = (u, v) in cycle_edges

            color = (
                "darkorange" if is_cycle
                else "crimson" if w < 0
                else "#8993a2"
            )

            ax.annotate(
                "",
                xy=pos[v],
                xytext=pos[u],
                arrowprops=dict(
                    arrowstyle="-|>",
                    color=color,
                    linewidth=2 if is_cycle else (
                        1.35 if w < 0 else 0.65
                    ),
                    alpha=0.85,
                    shrinkA=13,
                    shrinkB=13,
                    connectionstyle="arc3,rad=0.12"
                )
            )

            # ----------------------------------
            # DISPLAY EDGE WEIGHTS
            # ----------------------------------
            if n <= 8:
                x1, y1 = pos[u]
                x2, y2 = pos[v]

                dx = x2 - x1
                dy = y2 - y1

                length = math.hypot(dx, dy)

                # Move labels away from the middle
                # Opposite edges get different positions
                offset = 0.15

                label_x = (
                    (x1 + x2) / 2
                    - offset * dy / length
                )

                label_y = (
                    (y1 + y2) / 2
                    + offset * dx / length
                )

                ax.text(
                    label_x,
                    label_y,
                    f"{w:g}",
                    ha="center",
                    va="center",
                    fontsize=9,
                    zorder=6,
                    bbox=dict(
                        facecolor="white",
                        edgecolor="none",
                        alpha=0.9
                    )
                )

        # Draw vertices
        for u, (x, y) in pos.items():
            ax.scatter(
                x, y,
                color="#2563eb",
                s=400,
                zorder=4
            )

            ax.text(
                x, y, str(u),
                color="white",
                weight="bold",
                fontsize=9,
                ha="center",
                va="center",
                zorder=5
            )

        ax.axis("off")
        ax.set_aspect("equal")
        ax.set_xlim(-1.3, 1.3)
        ax.set_ylim(-1.3, 1.3)

        ax.set_title(
            "Graph: red = negative edge; "
            "orange = negative cycle"
        )

    # ----------------------------------
    # EDGE-WEIGHT MATRIX
    # ----------------------------------
    ax = next(axes)

    if show_matrix:
        matrix = np.full((n, n), np.nan)

        for u, v, w in edges:
            matrix[u, v] = w

        masked = np.ma.masked_invalid(matrix)

        cmap = plt.get_cmap("coolwarm").copy()
        cmap.set_bad("#eeeeee")

        image = ax.imshow(masked, cmap=cmap)

        fig.colorbar(
            image, ax=ax,
            fraction=0.046,
            label="weight"
        )

        ax.set_xlabel("destination")
        ax.set_ylabel("source")
        ax.set_title("Edge weights (grey = no edge)")

        if n <= 10:
            for i in range(n):
                for j in range(n):
                    if not np.isnan(matrix[i, j]):
                        ax.text(
                            j, i,
                            f"{matrix[i, j]:g}",
                            ha="center",
                            va="center",
                            fontsize=7
                        )

    else:
        ax.hist(
            [w for _, _, w in edges],
            bins=25,
            edgecolor="white"
        )

        ax.set_title("Edge weights (large graph)")
        ax.set_xlabel("weight")
        ax.set_ylabel("count")

    # ----------------------------------
    # EXPECTED SHORTEST DISTANCES
    # ----------------------------------
    if show_expected:
        ax = next(axes)

        distances = np.array(expected, dtype=float)
        masked = np.ma.masked_invalid(distances)

        image = ax.imshow(masked, cmap="coolwarm")

        fig.colorbar(
            image, ax=ax,
            fraction=0.046,
            label="shortest distance"
        )

        ax.set_title(
            "Expected all-pairs shortest distances"
        )

        ax.set_xlabel("destination")
        ax.set_ylabel("source")

        if n <= 10:
            for i in range(n):
                for j in range(n):
                    v = distances[i, j]

                    ax.text(
                        j, i,
                        "∞" if math.isinf(v) else f"{v:g}",
                        ha="center",
                        va="center",
                        fontsize=7
                    )

    # ----------------------------------
    # GRAPH TITLE
    # ----------------------------------
    label = (
        "NEGATIVE CYCLE (invalid for Johnson)"
        if expected == "NEGATIVE_CYCLE"
        else "No negative cycle expected"
    )

    fig.suptitle(
        f"{ds['name']} | {n} nodes, "
        f"{len(edges)} edges | {label}",
        fontsize=12
    )

    return fig


In [4]:
def ask_int(label, default, low=None, high=None):
    while True:
        value = input(f"{label} [default {default}]: ").strip()
        try:
            num = default if not value else int(value)
        except ValueError:
            print("Please enter a whole number.")
            continue
        if (low is not None and num < low) or (high is not None and num > high):
            print(f"Enter a value from {low} to {high}.")
            continue
        return num


def ask_float(label, default, low=None, high=None):
    while True:
        value = input(f"{label} [default {default}]: ").strip()
        try:
            num = default if not value else float(value)
        except ValueError:
            print("Please enter a number.")
            continue
        if (low is not None and num < low) or (high is not None and num > high):
            print(f"Enter a value from {low} to {high}.")
            continue
        return num


def ask_yes(label, default=True):
    while True:
        value = input(f"{label} [{'Y/n' if default else 'y/N'}]: ").strip().lower()
        if not value:
            return default
        if value in ("yes", "y"):
            return True
        if value in ("no", "n"):
            return False
        print("Please answer y or n.")


def ask_seed():
    while True:
        raw = input("Random seed [blank = NEW random graph]: ").strip()
        if not raw:
            return None
        try:
            return int(raw)
        except ValueError:
            print("Enter an integer seed or leave blank.")


def show_saved_dataset():
    """Load one saved file (including datasets generated from the terminal)."""
    candidates = sorted(DATA_DIR.rglob("*.txt"),
                        key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        print("No saved datasets yet. Generate one first.")
        return
    print("\nSaved datasets (newest first):")
    for ix, path in enumerate(candidates, 1):
        print(f"  {ix}) {path.relative_to(BASE_DIR)}")
    ix = ask_int("Dataset number", 1, 1, len(candidates))
    path = candidates[ix - 1]
    ds = gen.load_dataset(str(path))
    print(gen.describe(ds))
    show_and_save(ds, path.parent, save=False)


def show_and_save(ds, output_dir=None, save=True):
    """Save a newly generated dataset, then visualize it."""
    output_dir = Path(output_dir or (DATA_DIR / ds["mode"]))
    if save:
        saved = gen.save_dataset(ds, str(output_dir))
        print("Dataset saved:", saved)
    print(gen.describe(ds))
    query = 0
    if ds["mode"] == "astar" and len(ds["queries"]) > 1:
        for i, (s, t, d) in enumerate(ds["queries"], 1):
            print(f"  Query {i}: {s} -> {t}; expected {pretty_distance(d)}")
        query = ask_int("A* query to draw", 1, 1, len(ds["queries"])) - 1
    display_dataset(ds, query=query, save_png=True, out_dir=output_dir)


def generate_dataset_menu():
    """Random graphs and grids: always return to this submenu after a plot."""
    while True:
        print("\n===== GENERATE RANDOM DATASETS =====")
        print("1) Johnson — random weighted graph")
        print("2) Johnson — random graph with NEGATIVE cycle")
        print("3) A* — random grid (optional obstacles)")
        print("4) A* — random geometric graph")
        print("0) Back to main menu")
        choice = input("Choose dataset type [0-4]: ").strip()
        if choice in ("0", "", "back"):
            return
        if choice not in ("1", "2", "3", "4"):
            print("Choose a number from 0 to 4.")
            continue
        try:
            if choice in ("1", "2"):
                n = ask_int("Number of vertices", 10, 2, 5000)
                maximum = n * (n - 1)
                count = ask_int("Number of directed edges", min(30, maximum), 1, maximum)
                connected = count >= n and ask_yes("Force strongly connected?", True)
                ds = gen.johnson_dataset(n=n, edge_count=count, seed=ask_seed(),
                                         connected=connected, negative_cycle=(choice == "2"))
            elif choice == "3":
                rows = ask_int("Grid rows", 5, 2, 1000)
                cols = ask_int("Grid columns", 5, 2, 1000)
                obstacles = ask_float("Obstacle probability (0.0 to 0.6)", 0.2, 0, 0.6)
                r0 = ask_int("Start row", 0, 0, rows - 1)
                c0 = ask_int("Start column", 0, 0, cols - 1)
                r1 = ask_int("Goal row", rows - 1, 0, rows - 1)
                c1 = ask_int("Goal column", cols - 1, 0, cols - 1)
                ds = gen.grid_dataset(rows=rows, cols=cols, obstacle_prob=obstacles,
                                      seed=ask_seed(), start=(r0, c0), goal=(r1, c1))
            else:
                n = ask_int("Number of vertices", 60, 3, 5000)
                isolated = ask_yes("Isolate last node for unreachable-path test?", False)
                m = n - 1 if isolated else n
                max_extra = m * (m - 1) // 2 - (m - 1)
                extra = ask_int("Extra undirected edges", min(100, max_extra), 0, max_extra)
                queries = ask_int("Number of start-goal queries", 3, 1, 50)
                ds = gen.geometric_dataset(n=n, extra_edges=extra, seed=ask_seed(),
                                           isolate_last=isolated, num_queries=queries)
            show_and_save(ds)
        except (ValueError, OSError, AssertionError) as exc:
            print("Could not generate dataset:", exc)


def run_simulation():
    """One single entry point for the complete project."""
    while True:
        print("\n========== GRAPH SIMULATION ==========")
        print("1) Generate random datasets (Johnson / A*)")
        print("2) Run assignment TEST CASES (Johnson / A* vs Dijkstra)")
        print("3) Open and visualize a saved dataset")
        print("0) Exit")
        choice = input("Choose an option [0-3]: ").strip()
        if choice in ("0", "q", "quit", ""):
            print("Simulation menu closed.")
            return
        try:
            if choice == "1":
                generate_dataset_menu()
            elif choice == "2":
                assignment_test_menu()
            elif choice == "3":
                show_saved_dataset()
            else:
                print("Choose 0, 1, 2, or 3.")
        except (ValueError, OSError, AssertionError) as exc:
            print("Error:", exc)
            print("Back to main menu.")


In [5]:
def dijkstra_reference(ds, query=0):
    """Dijkstra with a priority queue; return distance and expanded node IDs."""
    start, goal, _ = ds["queries"][query]
    adj = {u: [] for u in ds["coords"]}
    for u, v, w in ds["edges"]:
        if w < 0:
            raise ValueError("Dijkstra requires nonnegative weights")
        adj[u].append((v, w))
    dist = {start: 0.0}
    expanded = set()
    pq = [(0.0, start)]
    while pq:
        current, u = heapq.heappop(pq)
        if u in expanded or current > dist.get(u, math.inf):
            continue
        expanded.add(u)
        if u == goal:
            return current, expanded
        for v, w in adj.get(u, []):
            nd = current + w
            if nd < dist.get(v, math.inf):
                dist[v] = nd
                heapq.heappush(pq, (nd, v))
    return math.inf, expanded


def plot_expansion_comparison(ds, astar_nodes, dijkstra_nodes, astar_path):
    """Maps of expanded nodes plus a simple bar chart for 100x100 benchmark."""
    coords = ds["coords"]
    w = int(max(x for x, y in coords.values())) + 1
    h = int(max(y for x, y in coords.values())) + 1
    open_cells = np.zeros((h, w))
    for x, y in coords.values():
        open_cells[int(y), int(x)] = 1
    fig, axes = plt.subplots(1, 3, figsize=(17, 5.5), layout="constrained")
    start, goal, _ = ds["queries"][0]
    for ax, nodes, title in ((axes[0], astar_nodes, "A* expanded"),
                             (axes[1], dijkstra_nodes, "Dijkstra expanded")):
        ax.imshow(open_cells, cmap="gray", origin="upper", vmin=0, vmax=1)
        overlay = np.zeros((h, w))
        for u in nodes:
            x, y = coords[u]
            overlay[int(y), int(x)] = 1
        ax.imshow(np.ma.masked_where(overlay == 0, overlay), cmap="Oranges",
                  alpha=0.65, origin="upper", vmin=0, vmax=1)
        if astar_path:
            ax.plot([coords[u][0] for u in astar_path],
                    [coords[u][1] for u in astar_path],
                    color="crimson", lw=1.8)
        ax.scatter(*coords[start], color="limegreen", s=75, edgecolor="black")
        ax.scatter(*coords[goal], color="gold", marker="*", s=95, edgecolor="black")
        ax.set_title(f"{title}: {len(nodes):,} nodes")
        ax.set_aspect("equal")
    axes[2].bar(["A*", "Dijkstra"], [len(astar_nodes), len(dijkstra_nodes)],
                color=["#16a34a", "#64748b"])
    axes[2].set_ylabel("Expanded nodes")
    axes[2].set_title("Node expansion comparison")
    fig.suptitle("Assignment Test 3 — A* vs Dijkstra (100 × 100 open grid)", fontsize=13)
    return fig


def run_johnson_assignment_test(case_number):
    ds = gen.fixed_cases()[case_number - 1]
    output_dir = DATA_DIR / "johnson"
    path = gen.save_dataset(ds, str(output_dir))
    print("\nJohnson assignment test", case_number)
    print("Saved:", path)
    if case_number == 1:
        required = [[0, -2, -3, -2],
                    [3, 0, -1, 0],
                    [4, 2, 0, 1],
                    [math.inf, math.inf, math.inf, 0]]
        assert ds["expected"] == required, "Reference matrix differs from required output"
        print("Expected shortest-distance matrix:")
        for row in required:
            print(" ", ["inf" if math.isinf(x) else x for x in row])
        print("Reference dataset check: PASS")
    else:
        assert ds["expected"] == "NEGATIVE_CYCLE"
        cycle = negative_cycle_nodes(len(ds["coords"]), ds["edges"])
        assert cycle, "Reference negative-cycle detector did not find a cycle"
        print("Expected: Negative Cycle Detected")
        print("Detected reference cycle:", " -> ".join(map(str, cycle + cycle[:1])))
        print("Reference dataset check: PASS")
    display_dataset(ds, save_png=True, out_dir=output_dir)
    print("Note: This checks the reference dataset, not your johnson.py implementation.")


def run_astar_assignment_test():
    print("\nA* vs Dijkstra — fixed 100 x 100 open grid, (0,0) to (99,99)")
    ds = gen.grid_dataset(100, 100, obstacle_prob=0.0, seed=0,
                          start=(0, 0), goal=(99, 99), name="astar_grid100_open")
    output_dir = DATA_DIR / "astar"
    path = gen.save_dataset(ds, str(output_dir))
    print("Dataset saved:", path)
    astar_dist, path_nodes, astar_expanded = astar_reference(ds)
    dijkstra_dist, dijkstra_expanded = dijkstra_reference(ds)
    expected = ds["queries"][0][2]
    assert astar_dist == dijkstra_dist == expected == 198, "Expected distance mismatch"
    reduction = 100 * (1 - len(astar_expanded) / len(dijkstra_expanded))
    print("A* distance:", astar_dist)
    print("Dijkstra distance:", dijkstra_dist)
    print("A* expanded nodes:", len(astar_expanded))
    print("Dijkstra expanded nodes:", len(dijkstra_expanded))
    print(f"A* expansion reduction: {reduction:.2f}%")
    print("Reference distance check: PASS")
    print("Note: Node counts depend on tie-breaking; these are reference implementations.")
    fig = plot_expansion_comparison(ds, set(astar_expanded), dijkstra_expanded, path_nodes)
    image_dir = output_dir / "images"
    image_dir.mkdir(parents=True, exist_ok=True)
    output_path = image_dir / "astar_grid100_open_comparison.png"
    fig.savefig(output_path, dpi=140, bbox_inches="tight")
    print("Comparison image saved:", output_path)
    plt.show()
    plt.close(fig)


def assignment_test_menu():
    while True:
        print("\n===== ASSIGNMENT TEST CASES =====")
        print("1) Johnson Test 1 — negative weights, no negative cycle")
        print("2) Johnson Test 2 — negative cycle detection")
        print("3) A* Test 3 — compare A* and Dijkstra on 100×100 grid")
        print("0) Back to main menu")
        choice = input("Choose test case [0-3]: ").strip()
        if choice in ("0", "", "back"):
            return
        try:
            if choice in ("1", "2"):
                run_johnson_assignment_test(int(choice))
            elif choice == "3":
                run_astar_assignment_test()
            else:
                print("Choose 0, 1, 2, or 3.")
        except (ValueError, OSError, AssertionError) as exc:
            print("Test could not complete:", exc)


In [ ]:
run_simulation()


========== GRAPH SIMULATION ==========
1) Generate random datasets (Johnson / A*)
2) Run assignment TEST CASES (Johnson / A* vs Dijkstra)
3) Open and visualize a saved dataset
0) Exit
